In [4]:
import pandas as pd
import os
import re
import numpy as np
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from scipy.spatial import KDTree

df = pd.read_csv("nyc_bus_cleaned.csv")

cols = [
    "PublishedLineName",
    "DirectionRef",
    "NextStopPointName",
    "VehicleLocation.Latitude",
    "VehicleLocation.Longitude",
    "ArrivalProximityText",
    "DistanceFromStop"
]

df = df[cols].dropna(subset=["PublishedLineName", "NextStopPointName"])

df_stops = df[
    df["ArrivalProximityText"].isin(["at stop", "approaching"])
].copy()

stops_per_line = (
    df_stops
    .groupby(["PublishedLineName", "DirectionRef", "NextStopPointName"], as_index=False)
    .agg({
        "VehicleLocation.Latitude": "median",
        "VehicleLocation.Longitude": "median"
    })
    .rename(columns={
        "VehicleLocation.Latitude": "stop_lat",
        "VehicleLocation.Longitude": "stop_lon"
    })
)

output_folder = "bus_lines_csv"
os.makedirs(output_folder, exist_ok=True)

def clean_filename(name):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(name))


for line, group in stops_per_line.groupby("PublishedLineName"):
    filename = clean_filename(line) + ".csv"
    path = os.path.join(output_folder, filename)

    group = group.sort_values(["DirectionRef", "NextStopPointName"])
    group.to_csv(path, index=False)

print("Done. CSV files saved in:", output_folder)

Done. CSV files saved in: bus_lines_csv


In [5]:
df_ev = pd.read_csv("NYC_Permitted_Event_Information_-_Historical_20260417.csv")
df_ev.head()


,Event ID,Event Name,Start Date/Time,End Date/Time,Event Agency,Event Type,Event Borough,Event Location,Event Street Side,Street Closure Type,Community Board,Police Precinct
0,317811,Public Menorah Lighting,01/01/2017 04:00:00 PM,01/01/2017 05:00:00 PM,Parks Department,Special Event,Manhattan,Sutton Place Park: Pig Plaza 57th,NaN,NaN,"6,","17,"
1,256697,Menorah Display,01/01/2017 04:00:00 PM,01/01/2017 08:00:00 PM,Parks Department,Special Event,Brooklyn,Brower Park (PS 289 - Playground Only): Lawn A...,NaN,NaN,"8,","77,"
2,320719,Nuestra Senora de La Nube,01/01/2017 04:00:00 PM,01/01/2017 05:00:00 PM,Police Department,Parade,Manhattan,"447 EAST 115 STREET, EAST 115 STREET between...",NaN,Full Street Closure,"11,","23, 25,"
3,310741,Menorah Lighting Ceremony,01/01/2017 04:00:00 PM,01/01/2017 05:00:00 PM,Parks Department,Special Event,Bronx,Bell Tower Park: Bell Tower Park,NaN,NaN,"8,","50,"
4,318588,Menorah Display and LIghting,01/01/2017 05:00:00 PM,01/01/2017 08:00:00 PM,Parks Department,Special Event,Manhattan,Bennett Park: Bennett Park,NaN,NaN,"12,","34,"


In [6]:
df_ev["Start Date/Time"] = pd.to_datetime(df_ev["Start Date/Time"], format="%m/%d/%Y %I:%M:%S %p")
df_ev["End Date/Time"] = pd.to_datetime(df_ev["End Date/Time"], format="%m/%d/%Y %I:%M:%S %p")

months_keep = [6, 8, 10, 12]

df_ev_filtered = df_ev[
    (df_ev["Start Date/Time"].dt.year == 2017) &
    (df_ev["Start Date/Time"].dt.month.isin(months_keep))
].copy()

df_ev_filtered["Start Date/Time"] = df_ev_filtered["Start Date/Time"].dt.strftime("%m/%d/%Y %I:%M:%S %p")
df_ev_filtered["End Date/Time"] = df_ev_filtered["End Date/Time"].dt.strftime("%m/%d/%Y %I:%M:%S %p")

df_ev_filtered.head()
df_ev_filtered.to_csv("nyc_events_filtered.csv", index=False)

In [7]:
df_bus = pd.read_csv("nyc_bus_cleaned.csv")
df_ev  = pd.read_csv("nyc_events_filtered.csv")

In [8]:
df_b41   = df_bus[df_bus["PublishedLineName"] == "B41"].copy()
df_stops = df_b41[df_b41["ArrivalProximityText"].isin(["at stop", "approaching"])].copy()

stops_b41 = (
    df_stops
    .groupby(["DirectionRef", "NextStopPointName"], as_index=False)
    .agg({"VehicleLocation.Latitude": "median", "VehicleLocation.Longitude": "median"})
    .rename(columns={
        "VehicleLocation.Latitude":  "stop_lat",
        "VehicleLocation.Longitude": "stop_lon",
    })
)
stops_b41

,DirectionRef,NextStopPointName,stop_lat,stop_lon
0,0.0,AV N/E 46 ST,40.618776,-73.929698
1,0.0,AV N/E 48 ST,40.618880,-73.928016
2,0.0,AV N/E 53 ST,40.619143,-73.923776
3,0.0,AV N/E 55 ST,40.619265,-73.921792
4,0.0,AV N/E 57 ST,40.619364,-73.920234
...,...,...,...,...
98,1.0,VETERANS AV/AV T,40.619858,-73.912038
99,1.0,VETERANS AV/E 64 ST,40.619624,-73.916070
100,1.0,VETERANS AV/E 66 ST,40.619752,-73.913877
101,1.0,VETERANS AV/E 69 ST,40.620009,-73.909791


In [9]:
def clean_event_location(loc):
    loc = str(loc).strip()
    loc = loc.split(",")[0].strip()
    loc = loc.split(":")[0].strip()
    return loc

df_ev = df_ev.dropna(subset=["Event Location"]).copy()
df_ev["location_clean"] = df_ev["Event Location"].apply(clean_event_location)

In [10]:
GEOCACHE_FILE = "geocache.csv"

unique_locs = df_ev["location_clean"].value_counts().reset_index().head(100)

if os.path.exists(GEOCACHE_FILE):
    geo_cache  = pd.read_csv(GEOCACHE_FILE)
    already    = set(geo_cache["location_clean"])
    to_geocode = unique_locs[~unique_locs["location_clean"].isin(already)].copy()
else:
    geo_cache  = pd.DataFrame(columns=["location_clean", "event_lat", "event_lon"])
    to_geocode = unique_locs.copy()

if not to_geocode.empty:
    geolocator = Nominatim(user_agent="bus_project", timeout=10)
    geocode_fn = RateLimiter(geolocator.geocode, min_delay_seconds=1.2)

    def geocode_loc(loc):
        try:
            res = geocode_fn(f"{loc}, Brooklyn, New York City, NY, USA")
            return pd.Series([res.latitude, res.longitude]) if res else pd.Series([np.nan, np.nan])
        except Exception:
            return pd.Series([np.nan, np.nan])

    to_geocode[["event_lat", "event_lon"]] = to_geocode["location_clean"].apply(geocode_loc)
    new_rows  = to_geocode[["location_clean", "event_lat", "event_lon"]].dropna()
    geo_cache = pd.concat([geo_cache, new_rows], ignore_index=True)
    geo_cache.to_csv(GEOCACHE_FILE, index=False)

geo_locs  = geo_cache.dropna(subset=["event_lat", "event_lon"])
df_ev_geo = df_ev.merge(geo_locs[["location_clean", "event_lat", "event_lon"]],
                        on="location_clean", how="inner")
print(f"Events with geocoded locations: {len(df_ev_geo)}")

RateLimiter caught an error, retrying (0/2 tries). Called with (*('PRINCE STREET between MOTT STREET and MULBERRY STREET, Brooklyn, New York City, NY, USA',), **{}).
Traceback (most recent call last):
  File "c:\Users\oierg\AppData\Local\Programs\Python\Python312\Lib\site-packages\geopy\geocoders\base.py", line 368, in _call_geocoder
    result = self.adapter.get_json(url, timeout=timeout, headers=req_headers)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\oierg\AppData\Local\Programs\Python\Python312\Lib\site-packages\geopy\adapters.py", line 472, in get_json
    resp = self._request(url, timeout=timeout, headers=headers)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\oierg\AppData\Local\Programs\Python\Python312\Lib\site-packages\geopy\adapters.py", line 500, in _request
    raise AdapterHTTPError(
geopy.adapters.AdapterHTTPError: Non-successful status code 429

The above exception was the direct cause 

Events with geocoded locations: 47817


In [11]:
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    a = (np.sin(np.radians(lat2 - lat1) / 2) ** 2
         + np.cos(phi1) * np.cos(phi2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2)
    return 2 * R * np.arcsin(np.sqrt(a))

stop_coords  = stops_b41[["stop_lat", "stop_lon"]].values
tree         = KDTree(stop_coords)
event_coords = df_ev_geo[["event_lat", "event_lon"]].values
_, idx       = tree.query(event_coords)

closest = stops_b41.iloc[idx].reset_index(drop=True)
df_ev_geo = df_ev_geo.reset_index(drop=True)
df_ev_geo["NextStopPointName"] = closest["NextStopPointName"].values
df_ev_geo["DirectionRef"]      = closest["DirectionRef"].values
df_ev_geo["distance_m"] = haversine_m(
    df_ev_geo["event_lat"].values, df_ev_geo["event_lon"].values,
    closest["stop_lat"].values,    closest["stop_lon"].values,
)

THRESHOLD = 800
df_ev_b41 = df_ev_geo[df_ev_geo["distance_m"] <= THRESHOLD].copy()
df_ev_b41.to_csv("B41_events_final.csv", index=False)
print(f"Events within {THRESHOLD} m: {len(df_ev_b41)}")

Events within 800 m: 5876


In [12]:
df_ev_b41["Start Date/Time"] = pd.to_datetime(df_ev_b41["Start Date/Time"])
df_ev_b41["End Date/Time"]   = pd.to_datetime(df_ev_b41["End Date/Time"])

YEAR        = 2017
MONTHS_KEEP = {6, 8, 10, 12}

df_ev_filt = df_ev_b41[
    (df_ev_b41["Start Date/Time"].dt.year  == YEAR) &
    (df_ev_b41["Start Date/Time"].dt.month.isin(MONTHS_KEEP))
].copy()

timeline = pd.date_range(f"{YEAR}-01-01", f"{YEAR}-12-31 23:30:00", freq="30min")
timeline = timeline[timeline.month.isin(MONTHS_KEEP)]

stops = df_ev_filt["NextStopPointName"].unique()

df_ts = pd.DataFrame(0, index=timeline, columns=stops, dtype=np.int8)
df_ts.index.name = "timestamp"

col_positions = {col: df_ts.columns.get_loc(col) for col in stops}

for _, ev in df_ev_filt.iterrows():
    stop = ev["NextStopPointName"]
    if stop not in col_positions:
        continue
    i_start = timeline.searchsorted(ev["Start Date/Time"])
    i_end   = timeline.searchsorted(ev["End Date/Time"], side="right")
    if i_start < i_end:
        df_ts.iloc[i_start:i_end, col_positions[stop]] = 1

df_out = df_ts.reset_index()
df_out.to_csv("B41_events_timeseries_30min.csv", index=False)
print(f"Saved: {df_out.shape[0]} rows x {df_out.shape[1]} columns")
df_out.head()

C:\Users\oierg\AppData\Local\Temp\ipykernel_33604\2392702255.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_ev_b41["End Date/Time"]   = pd.to_datetime(df_ev_b41["End Date/Time"])


Saved: 5904 rows x 6 columns


,timestamp,FLATBUSH AV/PROSPECT PARK ENT,LIVINGSTON ST/NEVINS ST,FLATBUSH AV/BERGEN ST,CADMAN PLZ W/MONTAGUE ST,FLATBUSH AV/PLAZA ST E
0,2017-06-01 00:00:00,0,0,0,0,0
1,2017-06-01 00:30:00,0,0,0,0,0
2,2017-06-01 01:00:00,0,0,0,0,0
3,2017-06-01 01:30:00,0,0,0,0,0
4,2017-06-01 02:00:00,0,0,0,0,0
